# ENEM 2024 · Sprint 6 — Ensemble, tuning de hiperparâmetros e SHAP

Fecha três itens da checklist de uma vez, e os três modelos que faltavam no **Plano, slide 10**
(*Ridge*, *Random Forest*, *LightGBM/CatBoost*):

| Item da checklist | Onde está aqui |
|---|---|
| **Ensemble** | seção 2 (Random Forest, boosting) e seção 5 (Voting e Stacking) |
| **Tuning de hiperparâmetros** | seção 3 (curva do Ridge) e seção 4 (`RandomizedSearchCV`) |
| **Interpretabilidade (SHAP)** | seção 6, com `permutation_importance` como alternativa sem dependência |

**Alvo:** `y_nota_ok`, o mesmo da Sprint 4, e as **mesmas 5 rodadas** (`KFold(5, shuffle=True,
random_state=0)`). Isso é o que torna os números comparáveis linha a linha com a referência linear —
comparar modelos em partições diferentes não quer dizer nada.

⚠️ **Só o conjunto de treino.** O teste continua guardado até a decisão final.

⏱️ **Custo:** o Random Forest e o Stacking são caros. Com `RAPIDO = True` o notebook roda em poucos
minutos para conferir se está tudo de pé; com `RAPIDO = False` ele roda a versão que vai para o G2.

## 0 · Setup

In [ ]:
import os, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import (HistGradientBoostingRegressor, RandomForestRegressor,
                              StackingRegressor, VotingRegressor)
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LinearRegression, Ridge, RidgeCV
from sklearn.model_selection import (KFold, RandomizedSearchCV, cross_validate,
                                     train_test_split, validation_curve)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

print("scikit-learn", sklearn.__version__)

PASTA  = os.environ.get("ENEM_OUT", "data_processed")
ARQ    = os.path.join(PASTA, "enem2024_features_ordinal.parquet")
ALVO   = "y_nota_ok"
SEED   = 42
RAPIDO = True                                     # False = versao do G2

AMOSTRA = 50_000 if RAPIDO else 150_000           # o Random Forest e o gargalo
N_ITER  = 8 if RAPIDO else 25                     # sorteios do RandomizedSearchCV
CV5     = KFold(5, shuffle=True, random_state=0)  # IDENTICA a da Sprint 4
CV3     = KFold(3, shuffle=True, random_state=0)  # busca interna, mais barata

pd.set_option("display.width", 170)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.alpha": .25})
AZUL, LARANJA, VERDE, CINZA = "#06406F", "#c2410c", "#15803d", "#9ca3af"

try:
    from lightgbm import LGBMRegressor
    TEM_LGBM = True
except ImportError:
    TEM_LGBM = False
print("LightGBM disponivel:", TEM_LGBM, "(se False, o HistGradientBoosting cobre o item do Plano)")

## 1 · Leitura

Idêntica à Sprint 4, para que a comparação seja justa. `te_municipio` e `te_uf` ficam **fora**: a
Sprint 4 (seção 3) mostrou que vazam na validação cruzada.

In [ ]:
FEATURES = pd.read_csv(os.path.join(PASTA, "lista_features.csv"))["feature"].tolist()
CAT = ["cor_raca", "regiao", "uf"]
TE  = ["te_municipio", "te_uf"]
NUM = [c for c in FEATURES if c not in CAT]
NUM_SEM_TE = [c for c in NUM if c not in TE]

base   = pd.read_parquet(ARQ, columns=["split", ALVO] + FEATURES)
treino = base[(base["split"] == "treino") & base[ALVO].notna()]
df = treino.sample(min(AMOSTRA, len(treino)), random_state=SEED).reset_index(drop=True)
del base, treino

X = df[FEATURES].copy()
X[NUM] = X[NUM].astype("float64")       # float32 derruba o R2 da linear (nota da Sprint 4)
X[CAT] = X[CAT].astype(str)
y = df[ALVO].astype("float64")

print(f"linhas: {len(X):,} | nota: media {y.mean():.1f} | desvio {y.std():.1f}")

## 2 · Os modelos candidatos (item *Ensemble*)

Um pré-processamento só serve a todos: `StandardScaler` nas numéricas e one-hot em `cor_raca` e `uf`.

- O **Ridge precisa** da padronização — sem ela o `alpha` penaliza cada coluna numa escala diferente
  e a regularização vira arbitrária.
- As **árvores não se importam** com escala, então padronizar não as prejudica. Um pipeline só, menos
  código para manter.
- `regiao` fica de fora: é função exata da UF (Sprint 4, seção 2).

**Random Forest** (bagging: muitas árvores fundas em amostras diferentes, média no fim) e
**boosting** (árvores rasas em sequência, cada uma corrigindo o erro da anterior) já **são** métodos de
ensemble — é o item da checklist. A seção 5 vai além e combina famílias diferentes.

In [ ]:
CAT_OHE = ["cor_raca", "uf"]
METRICAS = {"RMSE": "neg_root_mean_squared_error", "MAE": "neg_mean_absolute_error", "R2": "r2"}

def preparar():
    return ColumnTransformer([
        ("num", StandardScaler(), NUM_SEM_TE),
        ("cat", OneHotEncoder(drop="first", handle_unknown="ignore"), CAT_OHE),
    ])

def pipe(modelo):
    return Pipeline([("prep", preparar()), ("modelo", modelo)])

def avaliar(nome, modelo, X_=None, y_=None):
    # mesma CV de 5 partes da Sprint 4; devolve media e desvio de cada metrica
    t0 = time.time()
    r = cross_validate(modelo, X if X_ is None else X_, y if y_ is None else y_,
                       cv=CV5, scoring=METRICAS, n_jobs=-1)
    linha = {"modelo": nome}
    for m in METRICAS:
        v = r[f"test_{m}"] * (1 if m == "R2" else -1)   # o sklearn devolve erro com sinal negativo
        linha[m], linha[f"{m} dp"] = v.mean(), v.std()
    linha["seg"] = round(time.time() - t0, 1)
    print(f"  {nome:38} {linha['seg']:>7.1f}s")
    return linha

def tabela(linhas):
    t = pd.DataFrame(linhas).set_index("modelo")
    out = pd.DataFrame({m: t[m].map("{:.3f}".format) + " \u00b1 " + t[f"{m} dp"].map("{:.3f}".format)
                        for m in METRICAS})
    out["seg"] = t["seg"]
    return out

FLORESTA = RandomForestRegressor(n_estimators=100, min_samples_leaf=50, max_features="sqrt",
                                 random_state=SEED, n_jobs=-1)
BOOSTING = HistGradientBoostingRegressor(random_state=SEED)

candidatos = {
    "Chute: media do treino":        DummyRegressor(strategy="mean"),
    "Linear (referencia Sprint 4)":  pipe(LinearRegression()),
    "Ridge (alpha=1, padrao)":       pipe(Ridge(alpha=1.0, random_state=SEED)),
    "Random Forest":                 pipe(FLORESTA),
    "Boosting (HistGradient)":       pipe(BOOSTING),
}
if TEM_LGBM:
    candidatos["LightGBM"] = pipe(LGBMRegressor(random_state=SEED, verbose=-1))

print("rodando a CV de 5 partes:")
res = [avaliar(n, m) for n, m in candidatos.items()]
tabela(res)

**Como ler:** cada célula é **média ± desvio das 5 rodadas**. Um modelo só é melhor que a
referência linear se a diferença for **maior que as barras** — a frase da a15, "um R² sem o desvio
padrão é uma opinião", vale aqui mais que nunca, porque é tentador declarar vencedor por 0,002 de R².

Olhe também a coluna `seg`. Se o Random Forest custa 50× o tempo da linear para ganhar 0,01 de R², essa
é uma informação de decisão tão legítima quanto a métrica.

O Plano (slide 10) já avisava: **R² moderado é o esperado**, porque o η² de cada variável isolada não
passa de ~0,10 e a discordância entre corretores (correlação 0,78) limita o teto. Se algum modelo aqui
der R² muito alto, a primeira hipótese é vazamento, não genialidade.

## 3 · Tuning, parte 1: a curva de regularização do Ridge

A Sprint 4 (seção 5) diagnosticou **multicolinearidade**: `inse_proxy` é construído a partir de
`log_renda_pc`, `esc_max_pais` e `indice_bens`, então as quatro colunas dividem o crédito e os
coeficientes ficam instáveis. **Ridge é a resposta a esse diagnóstico** — ele encolhe os coeficientes
e distribui o peso entre colunas correlacionadas em vez de deixar o ajuste escolher arbitrariamente.

`alpha` é a força do encolhimento. Em vez de chutar, desenhamos a curva inteira.

In [ ]:
alphas = np.logspace(-2, 5, 24)
tr, te = validation_curve(pipe(Ridge(random_state=SEED)), X, y,
                          param_name="modelo__alpha", param_range=alphas,
                          cv=CV5, scoring="neg_root_mean_squared_error", n_jobs=-1)
rmse_tr, rmse_te = -tr.mean(axis=1), -te.mean(axis=1)
dp_te = te.std(axis=1)
melhor_alpha = alphas[rmse_te.argmin()]

fig, ax = plt.subplots(figsize=(7.2, 4.2))
ax.semilogx(alphas, rmse_tr, "o-", ms=3.5, color=CINZA, label="treino")
ax.semilogx(alphas, rmse_te, "o-", ms=3.5, color=AZUL, label="validacao (5 rodadas)")
ax.fill_between(alphas, rmse_te - dp_te, rmse_te + dp_te, color=AZUL, alpha=.15)
ax.axvline(melhor_alpha, ls="--", color=LARANJA, lw=1.3,
           label=f"melhor alpha = {melhor_alpha:.3g}")
ax.set_xlabel("alpha (forca da regularizacao)"); ax.set_ylabel("RMSE (pontos de nota)")
ax.set_title("Ridge: regularizacao de menos (esquerda) x de mais (direita)")
ax.legend(fontsize=8); plt.tight_layout(); plt.show()

res.append(avaliar(f"Ridge ajustado (alpha={melhor_alpha:.3g})",
                   pipe(Ridge(alpha=melhor_alpha, random_state=SEED))))
tabela(res)

**Como ler a curva:** à **esquerda** (alpha baixo) o Ridge vira a regressão linear — treino e
validação quase colados, porque com 65 features e dezenas de milhares de linhas este modelo não tem
capacidade para decorar. À **direita** (alpha alto) os coeficientes são espremidos a quase zero e o
modelo vira o chute da média: as duas curvas sobem juntas. É **underfitting**, não overfitting.

O fundo da curva azul é o alpha escolhido. Se a curva for quase plana numa faixa larga, **prefira o
alpha maior dentro do empate**: mesma precisão com coeficientes mais estáveis, que é exatamente o que
a Sprint 4 pedia para poder ler os pesos.

## 4 · Tuning, parte 2: busca aleatória no boosting

Para o modelo de árvores não dá para desenhar uma curva: são vários hiperparâmetros ao mesmo tempo.
Usamos **`RandomizedSearchCV`**, que sorteia `N_ITER` combinações, em vez de `GridSearchCV`, que testaria
o produto cartesiano inteiro.

**Por que aleatória e não grade:** com 5 hiperparâmetros e 4 valores cada, a grade são 1.024 ajustes.
A busca aleatória costuma achar uma combinação equivalente em uma fração disso, porque na prática só
dois ou três hiperparâmetros importam de verdade — e a grade gasta o mesmo esforço nos que não importam.

⚠️ A busca usa **CV de 3 partes internamente** (mais barata). O resultado da busca **não** é comparável
com a tabela: por isso o vencedor é reavaliado nas mesmas 5 rodadas de todo mundo, no fim da célula.

In [ ]:
espaco = {
    "modelo__max_iter":          [150, 300, 500],
    "modelo__learning_rate":     [0.03, 0.05, 0.1, 0.2],
    "modelo__max_leaf_nodes":    [15, 31, 63],
    "modelo__min_samples_leaf":  [20, 50, 100],
    "modelo__l2_regularization": [0.0, 0.1, 1.0],
}

busca = RandomizedSearchCV(pipe(HistGradientBoostingRegressor(random_state=SEED)),
                           param_distributions=espaco, n_iter=N_ITER, cv=CV3,
                           scoring="neg_root_mean_squared_error",
                           random_state=SEED, n_jobs=-1, refit=True)
t0 = time.time()
busca.fit(X, y)
print(f"{N_ITER} combinacoes em {time.time() - t0:.0f}s\n")
print("melhores hiperparametros:")
for k, v in busca.best_params_.items():
    print(f"  {k.replace('modelo__', ''):22} {v}")

ranking = (pd.DataFrame(busca.cv_results_)
           .nsmallest(5, "rank_test_score")[["rank_test_score", "mean_test_score", "std_test_score"]]
           .assign(RMSE=lambda d: -d["mean_test_score"])
           .rename(columns={"rank_test_score": "posicao", "std_test_score": "dp"})[["posicao", "RMSE", "dp"]])
display(ranking.round(3))

# reavalia o vencedor nas MESMAS 5 rodadas dos outros, para poder comparar
res.append(avaliar("Boosting ajustado", busca.best_estimator_))
tabela(res)

**Como ler o ranking:** compare o RMSE do 1º com o do 5º. Se a diferença couber dentro do `dp`,
o tuning **não achou nada** — os hiperparâmetros não são o que limita este problema, e o grupo pode
dizer isso com números em vez de insistir em buscas maiores. Esse é um resultado legítimo e, dado o
teto que o Plano previu, é o desfecho mais provável aqui.

## 5 · Ensemble de verdade: combinando famílias diferentes

Random Forest e boosting já são ensembles, mas **de árvores com árvores**. Combinar modelos que erram
do **mesmo jeito** não ajuda; o ganho vem de juntar famílias que erram de forma **diferente** — um
modelo linear e um de árvores.

Duas formas:

- **Voting** — média simples das previsões. Sem parâmetro para ajustar, sem risco de vazamento.
- **Stacking** — treina um **meta-modelo** (aqui, um `RidgeCV`) para aprender **quanto pesar** cada
  modelo base. É mais poderoso e mais caro: ele precisa de validação cruzada interna para gerar as
  previsões de treino do meta-modelo, senão vaza.

⏱️ O Stacking reajusta todos os modelos base várias vezes. É de longe a célula mais lenta do notebook.

In [ ]:
bases = [("ridge", pipe(Ridge(alpha=melhor_alpha, random_state=SEED))),
         ("floresta", pipe(FLORESTA)),
         ("boosting", pipe(HistGradientBoostingRegressor(random_state=SEED, **{
             k.replace("modelo__", ""): v for k, v in busca.best_params_.items()})))]

votacao = VotingRegressor(bases, n_jobs=-1)
empilha = StackingRegressor(bases, final_estimator=RidgeCV(), cv=CV3, n_jobs=-1)

print("rodando os ensembles (lento):")
res.append(avaliar("Ensemble: Voting (media)", votacao))
res.append(avaliar("Ensemble: Stacking (RidgeCV)", empilha))
tabela(res)

**Como ler:** se o ensemble empata com o melhor modelo isolado, a conclusão é que **os modelos
estão errando nas mesmas linhas** — não há diversidade para explorar. Num problema em que o teto é dado
pelo ruído do alvo (a discordância entre corretores), isso é o esperado, e vale escrever assim no G2.

Olhe os pesos que o Stacking aprendeu: se ele deu quase todo o peso a um modelo só, ele **descobriu
sozinho** que os outros não acrescentam.

In [ ]:
empilha.fit(X, y)
pesos = pd.Series(empilha.final_estimator_.coef_, index=[n for n, _ in bases])
print("peso que o meta-modelo deu a cada modelo base:")
print(pesos.round(3).to_string())
print(f"\nintercepto: {empilha.final_estimator_.intercept_:.1f}")

## 6 · Interpretabilidade (item *SHAP*)

Duas ferramentas, e elas respondem a perguntas **diferentes**:

| | Pergunta | Custo |
|---|---|---|
| **Permutation importance** | se eu **embaralhar** esta coluna, quanto o modelo piora? | barato, funciona em qualquer modelo |
| **SHAP** | para **esta pessoa**, quantos pontos cada feature somou ou tirou da previsão? | caro, mas dá o sinal e o caso individual |

Começamos pela permutation importance, que não precisa de dependência nova. Ela é medida num pedaço
**separado** (`validacao`), porque importância medida no próprio treino premia quem decorou.

⚠️ **Cuidado com features correlacionadas.** Embaralhar `log_renda_pc` sozinho não derruba muito o
modelo se `inse_proxy` e `renda_sm` continuam lá dizendo quase a mesma coisa. Ou seja: a importância
aparece **dividida** entre colunas redundantes, e nenhuma delas parece grande sozinha. É a mesma
multicolinearidade da seção 3, aparecendo de outro jeito.

In [ ]:
X_tr, X_val, y_tr, y_val = train_test_split(X, y, test_size=0.25, random_state=SEED)

modelo_final = pipe(HistGradientBoostingRegressor(random_state=SEED, **{
    k.replace("modelo__", ""): v for k, v in busca.best_params_.items()}))
modelo_final.fit(X_tr, y_tr)
print(f"R2 na validacao separada: {modelo_final.score(X_val, y_val):.3f}")

imp = permutation_importance(modelo_final, X_val, y_val, n_repeats=5,
                             random_state=SEED, n_jobs=-1, scoring="r2")
ordem = imp.importances_mean.argsort()[::-1][:15]

fig, ax = plt.subplots(figsize=(7, 5))
ax.barh([X.columns[i] for i in ordem][::-1], imp.importances_mean[ordem][::-1],
        xerr=imp.importances_std[ordem][::-1], color=AZUL, error_kw=dict(ecolor=CINZA, lw=1))
ax.set_xlabel("queda no R2 ao embaralhar a coluna")
ax.set_title("Permutation importance (15 maiores, medida fora do treino)")
plt.tight_layout(); plt.show()

### SHAP

`shap` não vem no Colab por padrão. Se a célula abaixo falhar, rode `!pip install shap` e repita —
e, se não quiser a dependência, a permutation importance acima **já cumpre** o item de
interpretabilidade da checklist.

Usamos o `TreeExplainer` sobre o **Random Forest**, porque o suporte do SHAP a florestas do sklearn é
o mais sólido, e sobre uma **amostra pequena**: o custo do SHAP exato em árvores cresce rápido.

In [ ]:
try:
    import shap
except ImportError:
    shap = None
    print("shap nao instalado. Rode:  !pip install shap   e execute esta celula de novo.")
    print("A permutation importance acima ja cobre o item de interpretabilidade.")

if shap is not None:
    N_SHAP = 1_000 if RAPIDO else 3_000
    rf = pipe(FLORESTA).fit(X_tr, y_tr)
    prep_rf, arvores_rf = rf.named_steps["prep"], rf.named_steps["modelo"]

    amostra = X_val.sample(min(N_SHAP, len(X_val)), random_state=SEED)
    Xt = prep_rf.transform(amostra)
    Xt = Xt.toarray() if hasattr(Xt, "toarray") else Xt
    nomes = [n.split("__", 1)[-1] for n in prep_rf.get_feature_names_out()]

    t0 = time.time()
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        valores = shap.TreeExplainer(arvores_rf).shap_values(Xt)
    print(f"SHAP em {len(amostra):,} linhas: {time.time() - t0:.0f}s")

    shap.summary_plot(valores, Xt, feature_names=nomes, max_display=15, show=False)
    plt.title("SHAP: efeito de cada feature na nota prevista", fontsize=11)
    plt.tight_layout(); plt.show()

**Como ler o summary plot:** cada ponto é **uma pessoa**. A posição horizontal é quantos **pontos
de nota** aquela feature somou (direita) ou tirou (esquerda) da previsão dela. A cor é o **valor** da
feature — vermelho alto, azul baixo.

Então "vermelho à direita" em `log_renda_pc` lê-se: *renda alta empurra a nota prevista para cima*. Uma
faixa **larga** quer dizer que a feature pesa muito em algumas pessoas e quase nada em outras — algo que
nem o coeficiente da Sprint 4 nem a permutation importance conseguem mostrar.

⚠️ **Isto continua sendo associação, não causa.** O SHAP explica **o modelo**, não o mundo: ele diz como
a previsão foi montada, não o que aconteceria se a renda de alguém mudasse. O confundidor da a03 segue
de pé.

## 7 · Resumo da Sprint 6

A célula abaixo monta o resumo a partir dos resultados que acabaram de rodar, inclusive o veredito
sobre trocar ou não a referência linear.

### Itens da checklist fechados aqui
- ✅ **Ensemble** — Random Forest, boosting, Voting e Stacking
- ✅ **Tuning de hiperparâmetros** — curva de validação do Ridge e `RandomizedSearchCV` no boosting
- ✅ **Interpretabilidade (SHAP)** — summary plot, com permutation importance de reserva
- ✅ **Plano, slide 10** — Ridge, Random Forest e LightGBM/boosting, os três que faltavam

### Ainda em aberto
- **Clustering** e **visualizações finais do G2**
- **Deploy (Streamlit)**
- Rodar o conjunto de **teste** — uma vez só, depois da decisão
- **Checklist ético e de viés** consolidado (as seções de equidade da S4 e da S5 são a matéria-prima)

In [ ]:
# --- resumo automatico: nada para preencher a mao ---
t = pd.DataFrame(res).set_index("modelo")
LINEAR = "Linear (referencia Sprint 4)"
melhor = t["R2"].idxmax()

ganho_r2 = t.loc[melhor, "R2"] - t.loc[LINEAR, "R2"]
barras   = t.loc[melhor, "R2 dp"] + t.loc[LINEAR, "R2 dp"]
top3 = [X.columns[i] for i in imp.importances_mean.argsort()[::-1][:3]]

print("RESUMO DA SPRINT 6")
print("=" * 64)
print(f"melhor modelo     : {melhor}")
print(f"                    RMSE {t.loc[melhor, 'RMSE']:.3f} +/- {t.loc[melhor, 'RMSE dp']:.3f}"
      f" | MAE {t.loc[melhor, 'MAE']:.3f} | R2 {t.loc[melhor, 'R2']:.3f} +/- {t.loc[melhor, 'R2 dp']:.3f}")
print(f"referencia linear : RMSE {t.loc[LINEAR, 'RMSE']:.3f} +/- {t.loc[LINEAR, 'RMSE dp']:.3f}"
      f" | R2 {t.loc[LINEAR, 'R2']:.3f} +/- {t.loc[LINEAR, 'R2 dp']:.3f}")
print(f"\nganho em R2 sobre a linear : {ganho_r2:+.4f}")
print(f"soma das barras de erro    : {barras:.4f}")
if ganho_r2 > barras:
    print("VEREDITO: o ganho PASSA das barras de erro -> trocar pelo modelo novo se valer o custo.")
else:
    print("VEREDITO: o ganho NAO passa das barras de erro -> a escolha honesta e ficar com a linear")
    print("          (mais barata, coeficiente legivel; o teto vem do ruido do alvo, nao do modelo).")

print(f"\nRidge             : melhor alpha {melhor_alpha:.3g}")
print(f"tuning            : {N_ITER} combinacoes; melhor RMSE da busca {-busca.best_score_:.3f}")
print(f"                    {busca.best_params_}")
print(f"ensembles         : " + " | ".join(
    f"{n.split(': ')[-1]} R2 {t.loc[n, 'R2']:.3f}" for n in t.index if n.startswith("Ensemble")))
print(f"pesos do stacking : {pesos.round(3).to_dict()}")
print(f"\ntop 3 features (permutation importance): {', '.join(top3)}")
print(f"custo             : melhor {t.loc[melhor, 'seg']}s vs linear {t.loc[LINEAR, 'seg']}s")
